# Quantum Information Science
### Qubits → Gates → Circuits → Entanglement → Algorithms
Every step shown. No gaps.

In [ ]:
from sympy import *
from sympy.physics.quantum import Ket, Bra, TensorProduct, Dagger
from sympy.physics.quantum.qubit import Qubit, measure_all
from sympy.physics.quantum.gate import *
from sympy.physics.quantum.qapply import qapply
from IPython.display import display, Markdown
init_printing(use_latex='mathjax')

def s(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None: display(expr)

def sec(t):  display(Markdown(f'---\n# {t}'))
def sub(t):  display(Markdown(f'## {t}'))

---
# §1 — The Qubit

In [ ]:
sec('The Qubit')
alpha, beta, theta, phi = symbols('alpha beta theta phi', real=True)

sub('Classical bit vs qubit')
s('Classical bit: 0 or 1  — two states, mutually exclusive')
s('Qubit: |ψ⟩ = α|0⟩ + β|1⟩   — superposition of both')
s('Constraint: |α|² + |β|² = 1  (normalization = total probability = 1)')
s('|α|² = probability of measuring 0')
s('|β|² = probability of measuring 1')

sub('Computational basis')
ket0 = Matrix([1, 0])
ket1 = Matrix([0, 1])
s('|0⟩ =', ket0)
s('|1⟩ =', ket1)
s('General qubit |ψ⟩ = α|0⟩ + β|1⟩ =')
psi_vec = alpha*ket0 + beta*ket1
s('', psi_vec)

sub('Bloch sphere parametrization')
s('Any qubit (up to global phase):')
s('|ψ⟩ = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩')
bloch = cos(theta/2)*ket0 + exp(I*phi)*sin(theta/2)*ket1
s('', bloch)
s('θ ∈ [0,π]: polar angle   (θ=0 → |0⟩, θ=π → |1⟩)')
s('φ ∈ [0,2π): azimuthal angle')
s('Poles: |0⟩ = north, |1⟩ = south')
s('Equator: equal superpositions — |+⟩, |−⟩, |i⟩, |−i⟩')

sub('Important states')
states = [
    ('|0⟩',  Matrix([1,0])),
    ('|1⟩',  Matrix([0,1])),
    ('|+⟩ = (|0⟩+|1⟩)/√2', Matrix([1,1])/sqrt(2)),
    ('|−⟩ = (|0⟩−|1⟩)/√2', Matrix([1,-1])/sqrt(2)),
    ('|i⟩ = (|0⟩+i|1⟩)/√2', Matrix([1,I])/sqrt(2)),
]
for name, vec in states:
    norm = sqrt((Dagger(Matrix(vec))*Matrix(vec))[0,0])
    s(f'{name}  norm={simplify(norm)}', vec)

---
# §2 — Single-Qubit Gates

In [ ]:
sec('Single-Qubit Gates')

sub('Pauli gates — the building blocks')
I2 = eye(2)
X  = Matrix([[0,1],[1,0]])      # NOT gate
Y  = Matrix([[0,-I],[I,0]])
Z  = Matrix([[1,0],[0,-1]])
H  = Matrix([[1,1],[1,-1]])/sqrt(2)   # Hadamard
S  = Matrix([[1,0],[0,I]])      # phase gate
T  = Matrix([[1,0],[0,exp(I*pi/4)]])  # T gate = π/8

gates = [('I (identity)', I2), ('X (NOT)',X), ('Y',Y), ('Z',Z),
         ('H (Hadamard)',H), ('S (phase)',S), ('T (π/8)',T)]

for name, G in gates:
    s(f'{name}:', G)
    herm = simplify(Dagger(G) - G)
    unitary = simplify(Dagger(G)*G - I2)
    s(f'  Hermitian (G†=G): {herm == zeros(2,2)}   Unitary (G†G=I): {unitary == zeros(2,2)}')

sub('Gate actions on basis states')
ket0 = Matrix([1,0])
ket1 = Matrix([0,1])
ketp = Matrix([1,1])/sqrt(2)
ketm = Matrix([1,-1])/sqrt(2)

s('X|0⟩ =', X*ket0,)
s('X|1⟩ =', X*ket1)
s('H|0⟩ =', simplify(H*ket0))
s('H|1⟩ =', simplify(H*ket1))
s('H|+⟩ =', simplify(H*ketp))
s('Z|+⟩ =', simplify(Z*ketp))
s('H·H = I:', simplify(H*H))

sub('Rotation gates — general qubit rotation')
t = symbols('t', real=True)
Rx = Matrix([[cos(t/2), -I*sin(t/2)],
             [-I*sin(t/2), cos(t/2)]])
Ry = Matrix([[cos(t/2), -sin(t/2)],
             [sin(t/2),  cos(t/2)]])
Rz = Matrix([[exp(-I*t/2), 0],
             [0, exp(I*t/2)]])
s('Rx(t) = exp(−iXt/2) =', Rx)
s('Ry(t) = exp(−iYt/2) =', Ry)
s('Rz(t) = exp(−iZt/2) =', Rz)
s('Verify Rx(π) = iX (up to global phase):', simplify(Rx.subs(t,pi)))
s('Verify Rz(π) = iZ:', simplify(Rz.subs(t,pi)))

---
# §3 — Two-Qubit Gates & Entanglement

In [ ]:
sec('Two-Qubit Gates & Entanglement')

sub('Tensor product — combining qubits')
ket0 = Matrix([1,0])
ket1 = Matrix([0,1])
s('|00⟩ = |0⟩⊗|0⟩ =')
ket00 = TensorProduct(ket0, ket0)
s('', Matrix([1,0,0,0]))
s('|01⟩ =', Matrix([0,1,0,0]))
s('|10⟩ =', Matrix([0,0,1,0]))
s('|11⟩ =', Matrix([0,0,0,1]))
s('Ordering: |q₁q₀⟩ — rightmost qubit is index 0')

sub('CNOT gate — controlled-NOT')
CNOT = Matrix([[1,0,0,0],
               [0,1,0,0],
               [0,0,0,1],
               [0,0,1,0]])
s('CNOT (control=q1, target=q0):', CNOT)
s('Action: flips target if control=|1⟩')
basis = [('|00⟩',Matrix([1,0,0,0])),('|01⟩',Matrix([0,1,0,0])),
         ('|10⟩',Matrix([0,0,1,0])),('|11⟩',Matrix([0,0,0,1]))]
for name, vec in basis:
    result = CNOT*vec
    s(f'CNOT|{name[1:-1]}⟩ =', result.T)

sub('Bell states — maximally entangled')
s('Create with H on control, then CNOT:')
H4 = TensorProduct(Matrix([[1,1],[1,-1]])/sqrt(2), eye(2))

bell_inputs = [
    ('|Φ+⟩ from |00⟩', Matrix([1,0,0,0])),
    ('|Φ-⟩ from |10⟩', Matrix([0,0,1,0])),
    ('|Ψ+⟩ from |01⟩', Matrix([0,1,0,0])),
    ('|Ψ-⟩ from |11⟩', Matrix([0,0,0,1])),
]
for name, inp in bell_inputs:
    bell = simplify(CNOT * H4 * inp)
    s(f'{name}:', bell.T)

sub('Why entanglement matters')
s('|Φ+⟩ = (|00⟩ + |11⟩)/√2')
s('Cannot be written as |ψ_A⟩⊗|ψ_B⟩ for any single-qubit states')
s('Measuring qubit A instantly determines qubit B — regardless of distance')
s('NOT faster-than-light signaling — no information transferred')
s('But correlations are stronger than any classical model can produce')

sub('Verify non-separability of |Φ+⟩')
a0,a1,b0,b1 = symbols('a0 a1 b0 b1')
sep = TensorProduct(Matrix([a0,a1]), Matrix([b0,b1]))
phi_plus = Matrix([1,0,0,1])/sqrt(2)
s('Separable state (a0,a1)⊗(b0,b1) =', sep)
s('Equations for |Φ+⟩:')
s('  a0*b0 = 1/√2,  a0*b1 = 0,  a1*b0 = 0,  a1*b1 = 1/√2')
s('  a0*b1=0 → a0=0 or b1=0')
s('  a0=0 → a0*b0=0 ≠ 1/√2  contradiction')
s('  b1=0 → a1*b1=0 ≠ 1/√2  contradiction')
s('  → NO separable solution exists  ∴ entangled')

---
# §4 — Quantum Circuits

In [ ]:
sec('Quantum Circuits')

sub('Circuit = sequence of unitary matrices (right to left = first to last)')
ket0 = Matrix([1,0])
ket1 = Matrix([0,1])
H  = Matrix([[1,1],[1,-1]])/sqrt(2)
X  = Matrix([[0,1],[1,0]])
Z  = Matrix([[1,0],[0,-1]])
S  = Matrix([[1,0],[0,I]])
T  = Matrix([[1,0],[0,exp(I*pi/4)]])

s('Circuit: H → X → H  applied to |0⟩')
result = H * X * H * ket0
s('= H·X·H|0⟩ =', simplify(result))
s('H·X·H = Z  (Hadamard conjugates X into Z):', simplify(H*X*H))

sub('Universality — any unitary from {H, T, CNOT}')
s('H + T generate all single-qubit rotations (dense in SU(2))')
s('Add CNOT → universal for all multi-qubit unitaries')
s('Proof sketch: any U ∈ SU(2^n) decomposes into two-level unitaries')
s('Each two-level unitary = sequence of CNOTs + single-qubit gates')

sub('T gate count = circuit complexity measure')
s('T gates are expensive on fault-tolerant hardware (require magic state distillation)')
s('T count of H: 0')
s('T count of S = T²: 2')
s('T count of Toffoli (CCX): 7')
Toffoli = zeros(8,8)
for i in range(6): Toffoli[i,i] = 1
Toffoli[6,7] = 1; Toffoli[7,6] = 1
s('Toffoli (CCX) matrix:', Toffoli)
s('Action: flips target iff BOTH controls = |1⟩')
s('Classical AND gate: Toffoli|a,b,0⟩ = |a,b,a AND b⟩')
s('→ quantum circuits can simulate classical logic reversibly')

---
# §5 — Quantum Algorithms

In [ ]:
sec('Quantum Algorithms')

sub('Deutsch-Jozsa — first quantum speedup')
s('Problem: f:{0,1}ⁿ→{0,1} is promised constant or balanced')
s('Classical: needs 2^(n-1)+1 queries worst case')
s('Quantum:   needs 1 query — always')
s('')
s('Circuit (n=1, Deutsch algorithm):')
s('1. Start |0⟩|1⟩')
s('2. Apply H⊗H: |+⟩|−⟩ = (|0⟩+|1⟩)/√2 · (|0⟩-|1⟩)/√2')
s('3. Apply U_f (oracle): |x⟩|y⟩ → |x⟩|y⊕f(x)⟩')
s('4. Apply H to first qubit')
s('5. Measure: |0⟩ = constant, |1⟩ = balanced')
s('')
s('Why it works — phase kickback:')
s('U_f|x⟩|−⟩ = (−1)^f(x)|x⟩|−⟩')
s('f(x) encoded in PHASE, not amplitude — H then reads the phase')

sub('Grover search — quadratic speedup')
N_s = symbols('N', positive=True, integer=True)
s('Problem: find marked item in unsorted database of N items')
s('Classical: O(N) queries average')
s('Quantum:   O(√N) queries')
s('')
s('Number of Grover iterations needed:')
grover_iters = pi/4 * sqrt(N_s)
s('  k ≈ π√N/4 =', grover_iters)
for n_val in [4, 16, 100, 1000000]:
    classical = n_val // 2
    quantum   = int(pi/4 * n_val**0.5)
    s(f'  N={n_val}: classical~{classical}, quantum~{quantum}')
s('')
s('Each iteration:')
s('  1. Oracle: flip phase of marked state |w⟩ → −|w⟩')
s('  2. Diffusion: inversion about average  2|ψ⟩⟨ψ| − I')
s('Geometrically: rotates state toward |w⟩ by angle 2arcsin(1/√N)')

sub('Quantum Fourier Transform (QFT)')
s('Classical DFT: O(N log N)')
s('QFT:          O((log N)²) — exponentially faster')
s('')
s('QFT|j⟩ = (1/√N) Σ_k exp(2πijk/N)|k⟩')
s('Matrix element: (QFT)_jk = (1/√N) ω^(jk)  where ω = e^(2πi/N)')
N_val = 4
omega = exp(2*pi*I/N_val)
QFT4 = Matrix([[omega**(j*k) for k in range(N_val)] for j in range(N_val)]) / sqrt(N_val)
s(f'QFT for N={N_val}:', simplify(QFT4))
s('Verify unitary (QFT†·QFT = I):', simplify(Dagger(QFT4)*QFT4))

sub('Shor\'s algorithm — exponential speedup for factoring')
s('Classical best: O(exp(n^(1/3))) — sub-exponential but not polynomial')
s('Shor:          O(n³) — polynomial in number of bits n')
s('')
s('Core insight: factoring reduces to PERIOD FINDING')
s('  Find r such that a^r ≡ 1 (mod N)')
s('  Then gcd(a^(r/2)±1, N) gives factors with high probability')
s('Period finding uses QFT — that\'s the quantum advantage')
s('')
s('Why this matters: RSA encryption relies on factoring being hard')
s('A 4000-qubit fault-tolerant quantum computer breaks RSA-2048')

---
# §6 — Density Matrices & Mixed States

In [ ]:
sec('Density Matrices — Noise & Mixed States')

sub('Pure vs mixed states')
s('Pure state: |ψ⟩ known exactly → ρ = |ψ⟩⟨ψ|')
s('Mixed state: classical uncertainty over pure states → ρ = Σᵢ pᵢ|ψᵢ⟩⟨ψᵢ|')
s('')
s('Properties of ρ:')
s('  ρ† = ρ          (Hermitian)')
s('  Tr(ρ) = 1       (normalized)')
s('  ρ ≥ 0           (positive semidefinite)')
s('  Tr(ρ²) = 1 pure,  Tr(ρ²) < 1 mixed')

ket0 = Matrix([1,0])
ket1 = Matrix([0,1])
ketp = Matrix([1,1])/sqrt(2)

sub('Examples')
rho_pure = ketp * ketp.T
s('|+⟩⟨+| (pure):', simplify(rho_pure))
s('Tr(ρ²) =', simplify((rho_pure**2).trace()))

rho_mixed = Rational(1,2)*ket0*ket0.T + Rational(1,2)*ket1*ket1.T
s('50/50 mixed ρ = (|0⟩⟨0| + |1⟩⟨1|)/2:', rho_mixed)
s('Tr(ρ²) =', simplify((rho_mixed**2).trace()))
s('= maximally mixed state — no quantum coherence')

sub('Decoherence — why quantum computers are hard')
s('Environment entangles with qubit → coherence (off-diagonals) decay')
gamma, t_s = symbols('Gamma t', positive=True)
rho_decay = Matrix([
    [1, exp(-gamma*t_s/2)],
    [exp(-gamma*t_s/2), 0]
])
s('Off-diagonal decay: ρ₀₁(t) = ρ₀₁(0)·e^(−Γt/2)')
s('ρ(t) =', rho_decay)
s('As t→∞: off-diagonals → 0 = classical mixture')
s('Decoherence time T₂ = 1/Γ must be >> gate time')
s('Current best: T₂ ~ ms, gate time ~ ns → ~10⁶ gates before decoherence')

---
# §7 — Connection to Optics & Your Project

In [ ]:
sec('Connection: Quantum Info ↔ Optics ↔ GS Phase Recovery')

sub('Optical qubits')
s('Polarization qubit: |H⟩ = |0⟩, |V⟩ = |1⟩')
s('Path qubit:         |upper⟩ = |0⟩, |lower⟩ = |1⟩')
s('Time-bin qubit:     |early⟩ = |0⟩, |late⟩ = |1⟩')
s('')
s('Gates in optics:')
s('  H gate → beamsplitter (50/50)')
s('  Z gate → half-wave plate (HWP) at 0°')
s('  X gate → HWP at 45°')
s('  Phase gate S → quarter-wave plate')

sub('Measurement = phase retrieval')
s('In QM: measuring |ψ⟩ collapses to eigenstate — phase lost')
s('In optics: detector measures |E(t)|² — phase lost')
s('GS algorithm: recover phase from intensity measurements')
s('  ↔ quantum state tomography: recover ρ from measurement statistics')
s('')
s('Both problems:')
s('  Known: magnitudes (intensities)')
s('  Unknown: phases')
s('  Method: alternating projections onto constraint sets')

sub('Dispersive diversity ↔ informationally complete measurements')
s('TDGSA: two measurements (D=0, D≠0) give enough info to recover phase')
s('Quantum tomography: needs informationally complete POVM')
s('  POVM = {Mᵢ}: Σᵢ Mᵢ†Mᵢ = I (completeness)')
s('  Each Mᵢ = one measurement setting')
s('')
s('Minimum measurements to fully characterize:')
s('  1 qubit:  4 measurements (I, X, Y, Z expectation values)')
s('  n qubits: 4ⁿ measurements (exponential — hard to scale)')
s('  Optical pulse: 2 intensity measurements + GS → tractable')

sub('Wigner function — phase space')
x_s, p_s = symbols('x p', real=True)
hbar = symbols('hbar', positive=True)
s('Wigner function W(x,p) = quasi-probability distribution in phase space')
s('W(x,p) = (1/πħ) ∫ ψ*(x+y)ψ(x−y) e^(2ipy/ħ) dy')
s('Can be negative — quantum signature (no classical analogue)')
s('Marginals give true probabilities:')
s('  ∫W(x,p)dp = |ψ(x)|²   (position distribution)')
s('  ∫W(x,p)dx = |φ(p)|²   (momentum distribution)')
s('Optical analogue: Wigner function of pulse field E(t)')
s('  ↔ spectrogram / FROG trace')
s('  Phase retrieval from FROG = recover W from its marginals')